# Table 1 – Stroke (demand code 54)
Descriptive statistics stratified by `p1_real_emerg` (real priority 1 assigned by the emergency service).

# Environment

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

sys.path.insert(0, os.path.abspath('..'))
from kbase.config import settings

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_info_columns', 50)

# warnings.filterwarnings('ignore')

# Data Loading

In [ ]:
df_triaje = pq.read_table(
    os.path.join(settings.source_tables_path, settings.stroke_table_cleaned_path)
).to_pandas()

if "demand_type_1" in df_triaje.columns:
    df_stroke_p1_assigned = df_triaje[df_triaje["demand_type_1"] == 54].copy()
else:
    df_stroke_p1_assigned = df_triaje.copy()
df_stroke_p1_real_emerg = df_stroke_p1_assigned.dropna(subset=["p1_real_emerg"])

# print(df_stroke_p1_assigned["p1_assigned"].value_counts())
# print(df_stroke_p1_real_emerg["p1_real_emerg"].value_counts(dropna=False))

print(f"Shape of dataframe with p1_real_emerg as outcome: {df_triaje.shape}")
df_triaje.head(3)

In [ ]:
df_triaje.info(max_cols=sys.maxsize)

# Parameters

In [ ]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
DATA_PATH = os.path.join(settings.source_tables_path, settings.stroke_table_cleaned_path)

# Columns needed only for identifiers, filtering, or provenance. They remain in
# df_triaje for date filtering, but are excluded from the default Table 1 rows.
COLS_TO_DROP = ["demandpk", "demand_date", "demand_type_1", "has_icd_emerg", "has_history"]
TABLE_COLUMNS = [col for col in df_triaje.columns if col not in COLS_TO_DROP]

VARS_NUMERIC = ["age"]
VARS_CATEGORICAL = ["year"]  # shown as N (%) per unique value, not as a numeric summary
VARS_BINARY = [
    col for col in [
        "sex", "triage", "p1_assigned", "p1_real_emerg",
    ]
    if col in TABLE_COLUMNS
]

GROUPS = {
    "Entity receiving alert": [col for col in TABLE_COLUMNS if col.startswith("alert_receiver_")],
    "Patient location": [col for col in TABLE_COLUMNS if col.startswith("location_patient_")],
    "Day of week": [col for col in TABLE_COLUMNS if col.startswith("day_week_")],
    "Time of day": [col for col in TABLE_COLUMNS if col.startswith("time_of_day_")],
    "Month": [col for col in TABLE_COLUMNS if col.startswith("month_")],
    "Province": [col for col in TABLE_COLUMNS if col.startswith("province_")],
    "Triage protocol questions": [
        col for col in TABLE_COLUMNS
        if col[0].isdigit() and col[-1].isalpha()
    ],
    "Literal reason features": [col for col in TABLE_COLUMNS if col.startswith("lr_")],
    "Medication groups": [col for col in ["has_med"] if col in TABLE_COLUMNS] + [col for col in TABLE_COLUMNS if col.startswith("atc_group_")],
    "Comorbidities and history": [col for col in ["has_com"] if col in TABLE_COLUMNS] + [col for col in TABLE_COLUMNS if col.startswith("com_")],
}
GROUPS = {group: cols for group, cols in GROUPS.items() if cols}


def _label_from_suffix(column: str, prefix: str, label_prefix: str = "") -> str:
    label = column.removeprefix(prefix).replace("_", " ").title()
    return f"{label_prefix}{label}"


def _label_from_coded_suffix(column: str, prefix: str, label_prefix: str = "") -> str:
    parts = column.removeprefix(prefix).split("_")
    label_parts = [part for part in parts if not part.isdigit()]
    label = " ".join(label_parts).title()
    return f"{label_prefix}{label}"


LABELS_MAP = {
    "age": "Age (years)",
    "year": "Year of the call",
    "sex": "Sex (male)",
    "triage": "Structured triage performed",
    "p1_assigned": "Assigned P1 priority",
    "p1_real_emerg": "Real emergency P1 priority",
    "has_icd_emerg": "Emergency ICD diagnosis available",
    "has_history": "Past medical history available",
    "has_med": "Medication history available",
    "has_com": "Comorbidity",
    "alert_receiver_112": "112 emergency line",
    "alert_receiver_user": "User or patient",
    "alert_receiver_pol_fg": "Police or Civil Guard",
    "alert_receiver_hs": "Health services",
    "alert_receiver_tele": "Tele-assistance",
    "alert_receiver_others": "Other alert receiver",
    "location_patient_home": "Patient at home",
    "location_patient_public_road": "Patient on public road",
    "location_patient_other": "Other patient location",
    "time_of_day_early_morning": "Early morning (00-05 h)",
    "time_of_day_morning": "Morning (06-11 h)",
    "time_of_day_afternoon": "Afternoon (12-17 h)",
    "time_of_day_night": "Night (18-23 h)",
}
LABELS_MAP.update({col: _label_from_suffix(col, "day_week_") for col in TABLE_COLUMNS if col.startswith("day_week_")})
LABELS_MAP.update({col: _label_from_suffix(col, "month_") for col in TABLE_COLUMNS if col.startswith("month_")})
LABELS_MAP.update({col: _label_from_suffix(col, "province_") for col in TABLE_COLUMNS if col.startswith("province_")})

# --- Triage protocol questions (A54 GCTT, questions P1-P6) ---
TRIAGE_LABELS = {
    "1a": "Q1A Eye opening - Yes",
    "1b": "Q1B Eye opening - No",
    "2a": "Q2A Respiration - Normal",
    "2b": "Q2B Respiration - Difficulty",
    "3a": "Q3A Complaint - Facial droop",
    "3b": "Q3B Complaint - Limb paralysis/weakness",
    "3c": "Q3C Complaint - Paresthesia",
    "3d": "Q3D Complaint - Speech difficulty",
    "3e": "Q3E Complaint - Seizures",
    "3f": "Q3F Complaint - Sudden vision loss",
    "4a": "Q4A First episode - Yes / prior without sequelae",
    "4b": "Q4B First episode - Prior with sequelae",
    "6a": "Q6A Functional status - Independent",
    "6b": "Q6B Functional status - Bedridden/dependent",
}
LABELS_MAP.update({col: TRIAGE_LABELS[col] for col in TABLE_COLUMNS if col in TRIAGE_LABELS})

LABELS_MAP.update({col: _label_from_suffix(col, "lr_") for col in TABLE_COLUMNS if col.startswith("lr_")})

LABELS_MAP.update({col: _label_from_suffix(col, "lr_") for col in TABLE_COLUMNS if col.startswith("lr_")})
LABELS_MAP.update({col: _label_from_suffix(col, "atc_group_") for col in TABLE_COLUMNS if col.startswith("atc_group_")})
LABELS_MAP.update({col: _label_from_coded_suffix(col, "com_") for col in TABLE_COLUMNS if col.startswith("com_")})

TITLE = "Table 1. Descriptive statistics of stroke calls stratified by real priority (p1_real_emerg)"

OUTPUT_DIR = os.path.join("figures", "04_stroke/tables")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Filtering options (used in build_table1) ---
SUBTABLE_NAME   = None          # e.g. "general" → files saved as table1_stroke_general.*
TABLE1_FILTERED = False         # master switch to enable date / variable filtering
YEARS_FILTERED  = None          # tuple of (start_date, end_date) strings, e.g. ("2020-03-15", "2021-12-31")
VARS_CONCRETE   = None          # list of column names to include; None = include all


# Functions

In [ ]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def _format_median_iqr(series: pd.Series) -> str:
    s = series.dropna().astype(float)
    if s.empty:
        return ""
    return f"{np.median(s):.1f} [{np.percentile(s, 25):.1f}, {np.percentile(s, 75):.1f}]"


def _fmt_pct(n, d) -> str:
    if d == 0:
        return "0 (0.0%)"
    return f"{int(n)} ({100.0 * n / d:.1f}%)"


def _smd_continuous(x0: pd.Series, x1: pd.Series) -> str:
    x0, x1 = x0.dropna().astype(float), x1.dropna().astype(float)
    pooled_sd = np.sqrt((x0.var(ddof=1) + x1.var(ddof=1)) / 2)
    if pooled_sd == 0:
        return "NA"
    smd = abs(x1.mean() - x0.mean()) / pooled_sd
    se  = np.sqrt((len(x0) + len(x1)) / (len(x0) * len(x1)) + smd**2 / (2 * (len(x0) + len(x1))))
    return f"{smd:.2f} [{smd - 1.96*se:.2f}, {smd + 1.96*se:.2f}]"


def _smd_binary(p0: float, p1: float, n0: int, n1: int) -> str:
    denom = np.sqrt((p0 * (1 - p0) + p1 * (1 - p1)) / 2)
    if denom == 0:
        return "NA"
    smd = abs(p1 - p0) / denom
    se  = np.sqrt(1 / (2 * n0) + 1 / (2 * n1))
    return f"{smd:.2f} [{smd - 1.96*se:.2f}, {smd + 1.96*se:.2f}]"


def _pvalue_str(p: float) -> str:
    return "<0.001" if p < 0.001 else f"{p:.3f}"


In [ ]:
# ---------------------------------------------------------------------------
# Main function
# ---------------------------------------------------------------------------

def build_table1(
    data_path:        str,
    outcome:          str,
    vars_numeric:     list,
    vars_categorical: list,
    vars_binary:      list,
    groups:           dict,
    labels_map:       dict,
    title:            str,
    output_dir:       str,
    subtable_name:    str          = None,
    table1_filtered:  bool         = False,
    years_filtered:   tuple        = None,
    vars_concrete:    list         = None,
    df_input:         pd.DataFrame = None,
    outcome_label:    str          = None,
) -> pd.DataFrame:
    """
    Build Table 1 stratified by a binary outcome.

    Numeric vars      → median [Q1, Q3] | Mann-Whitney U | SMD continuous
    Categorical vars  → header with omnibus chi-square p-value + one N (%) row per unique value
    Binary vars       → N (%)           | Chi-square / Fisher | SMD proportions
    Grouped vars      → group header row + one N(%) row per dummy

    Parameters
    ----------
    subtable_name : str, optional
        Suffix appended to the output file names, e.g. "general" produces
        table1_stroke_general.docx / .jpg. Default None keeps table1_stroke.*.
    table1_filtered : bool
        Master switch. When True, applies date and/or variable filters.
    years_filtered : tuple of (str, str), optional
        (start_date, end_date) in "YYYY-MM-DD" format. Filters rows by
        demand_date when table1_filtered=True.
    vars_concrete : list, optional
        Flat list of column names to include. Just name the columns you want
        — the function assigns each one to the correct section automatically.
        Applied only when table1_filtered=True. Default None includes all variables.
    df_input : pd.DataFrame, optional
        Required when outcome='p1_assigned'. The pre-filtered DataFrame to use
        instead of reading from data_path.
    outcome_label : str, optional
        Display name for the stratifying outcome, used in the group column
        headers (e.g. "P1 (real emergency)"). Default None falls back to
        the raw `outcome` column name.
    """

    # --- Load ---
    if outcome == "p1_real_emerg":
        df = pq.read_table(data_path).to_pandas()
    elif outcome == "p1_assigned":
        if df_input is None:
            raise ValueError("df_input is required when outcome='p1_assigned'")
        df = df_input

    if outcome not in df.columns:
        raise ValueError(f"Outcome column '{outcome}' not found.")

    vars_numeric = [v for v in vars_numeric if v != outcome]
    vars_categorical = [v for v in vars_categorical if v != outcome]
    vars_binary = [v for v in vars_binary if v != outcome]
    groups = {
        group: [v for v in members if v != outcome]
        for group, members in groups.items()
    }

    # --- Date filter first (before outcome filter, so counts are meaningful) ---
    if table1_filtered and years_filtered is not None:
        if "demand_date" not in df.columns:
            raise ValueError(
                "years_filtered requires 'demand_date' in the dataset, "
                "but the column was not found. Load a parquet that includes it."
            )
        start, end = pd.Timestamp(years_filtered[0]), pd.Timestamp(years_filtered[1])
        df = df.loc[df["demand_date"].between(start, end)].copy()

    # --- Filter to valid binary outcome values ---
    n_total_before  = len(df)
    outcome_counts  = df[outcome].value_counts(dropna=False).to_dict()
    outcome_dtype   = df[outcome].dtype
    df = df.loc[df[outcome].isin([0, 1])].copy()

    if len(df) == 0:
        period_info = (
            f" in the period {years_filtered[0]} – {years_filtered[1]}"
            if years_filtered else ""
        )
        raise ValueError(
            f"No rows with '{outcome}' ∈ {{0, 1}} found{period_info}.\n"
            f"  rows in period : {n_total_before:,}\n"
            f"  dtype          : {outcome_dtype}\n"
            f"  value counts   : {outcome_counts}\n"
            f"Possible causes: values outside this period differ from the full dataset "
            f"(e.g. NaN, -1), or the dtype is string so integer isin() does not match."
        )

    # --- Variable filters ---
    if table1_filtered and vars_concrete is not None:
        vars_numeric     = [v for v in vars_numeric     if v in vars_concrete]
        vars_categorical = [v for v in vars_categorical if v in vars_concrete]
        vars_binary      = [v for v in vars_binary      if v in vars_concrete]
        groups = {
            k: [v for v in vlist if v in vars_concrete]
            for k, vlist in groups.items()
            if any(v in vars_concrete for v in vlist)
        }

        # Classify vars_concrete variables not covered by any existing list
        already_covered = (
            set(vars_numeric)
            | set(vars_categorical)
            | set(vars_binary)
            | {v for vlist in groups.values() for v in vlist}
        )
        for v in vars_concrete:
            if v in already_covered or v not in df.columns:
                continue
            col = df[v].dropna()
            unique_vals = col.unique()
            if set(unique_vals).issubset({0, 1, 0.0, 1.0}):
                vars_binary.append(v)
            elif col.dtype.kind in ("i", "u", "f") and col.nunique() > 10:
                vars_numeric.append(v)
            else:
                vars_categorical.append(v)

    # --- Build display title with period / subtable info when filtered ---
    display_title = title
    if table1_filtered:
        suffix_parts = []
        if subtable_name is not None:
            suffix_parts.append(f"for {subtable_name}")
        if years_filtered is not None:
            suffix_parts.append(f"in the period between {years_filtered[0]} and {years_filtered[1]}")
        if suffix_parts:
            display_title = title + " " + " ".join(suffix_parts)

    g0, g1        = df[df[outcome] == 0], df[df[outcome] == 1]
    n_all, n0, n1 = len(df), len(g0), len(g1)

    strat_label = outcome_label if outcome_label is not None else outcome
    col_overall = f"Overall (N={n_all:,})"
    col_g0      = f"{strat_label}=0 (N={n0:,}, {100.0*n0/n_all:.1f}%)"
    col_g1      = f"{strat_label}=1 (N={n1:,}, {100.0*n1/n_all:.1f}%)"
    col_smd     = "SMD [95% CI]"
    col_p       = "p-value"

    all_vars_in_order = (
        vars_numeric
        + vars_categorical
        + vars_binary
        + [v for vlist in groups.values() for v in vlist]
    )
    missing = [v for v in all_vars_in_order if v not in df.columns]
    if missing:
        print(f"[WARNING] Variables not found and skipped: {missing}")

    rows      = []
    row_types = []

    def _add_row(label, overall, c0, c1, smd, p, rtype):
        rows.append({
            "Variable": label,
            col_overall: overall,
            col_g0:      c0,
            col_g1:      c1,
            col_smd:     smd,
            col_p:       p,
        })
        row_types.append(rtype)

    # ── Numeric variables ────────────────────────────────────────────────────
    for v in vars_numeric:
        if v not in df.columns:
            continue
        x0, x1 = g0[v], g1[v]
        _, pval = stats.mannwhitneyu(
            x0.dropna().astype(float), x1.dropna().astype(float), alternative="two-sided"
        )
        _add_row(
            label   = labels_map.get(v, v),
            overall = _format_median_iqr(df[v]),
            c0      = _format_median_iqr(x0),
            c1      = _format_median_iqr(x1),
            smd     = _smd_continuous(x0, x1),
            p       = _pvalue_str(pval),
            rtype   = "numeric",
        )

    # ── Categorical variables (e.g. year) ────────────────────────────────────
    for v in vars_categorical:
        if v not in df.columns:
            continue
        unique_vals = sorted(df[v].dropna().unique())
        contingency = pd.crosstab(df[v], df[outcome])
        _, pval_cat, _, _ = stats.chi2_contingency(contingency)
        _add_row(labels_map.get(v, v), "", "", "", "", _pvalue_str(pval_cat), rtype="header")
        for val in unique_vals:
            n_all_v = int((df[v] == val).sum())
            n0_v    = int((g0[v] == val).sum())
            n1_v    = int((g1[v] == val).sum())
            p0_v    = n0_v / n0
            p1_v    = n1_v / n1
            label_v = "  " + (str(int(val)) if isinstance(val, float) and val.is_integer() else str(val))
            _add_row(
                label   = label_v,
                overall = _fmt_pct(n_all_v, n_all),
                c0      = _fmt_pct(n0_v, n0),
                c1      = _fmt_pct(n1_v, n1),
                smd     = _smd_binary(p0_v, p1_v, n0, n1),
                p       = "",
                rtype   = "binary",
            )

    # ── Standalone binary variables ──────────────────────────────────────────
    for v in vars_binary:
        if v not in df.columns:
            continue
        n_all_1 = int((df[v] == 1).sum())
        n0_1    = int((g0[v] == 1).sum())
        n1_1    = int((g1[v] == 1).sum())
        p0, p1  = n0_1 / n0, n1_1 / n1

        tbl = np.array([[n1_1, n1 - n1_1], [n0_1, n0 - n0_1]])
        if tbl.min() < 5:
            _, pval = stats.fisher_exact(tbl)
        else:
            _, pval, _, _ = stats.chi2_contingency(tbl)

        _add_row(
            label   = labels_map.get(v, v),
            overall = _fmt_pct(n_all_1, n_all),
            c0      = _fmt_pct(n0_1, n0),
            c1      = _fmt_pct(n1_1, n1),
            smd     = _smd_binary(p0, p1, n0, n1),
            p       = _pvalue_str(pval),
            rtype   = "header",
        )

    # ── Grouped one-hot variables ────────────────────────────────────────────
    for group_label, members in groups.items():
        present = [v for v in members if v in df.columns]
        if not present:
            continue

        lead_vars = [v for v in ["has_med", "has_com"] if v in present]
        present = [v for v in present if v not in lead_vars]

        for v in lead_vars:
            n_all_1 = int((df[v] == 1).sum())
            n0_1    = int((g0[v] == 1).sum())
            n1_1    = int((g1[v] == 1).sum())
            p0, p1  = n0_1 / n0, n1_1 / n1

            tbl = np.array([[n1_1, n1 - n1_1], [n0_1, n0 - n0_1]])
            if tbl.min() < 5:
                _, pval = stats.fisher_exact(tbl)
            else:
                _, pval, _, _ = stats.chi2_contingency(tbl)

            _add_row(
                label   = labels_map.get(v, v),
                overall = _fmt_pct(n_all_1, n_all),
                c0      = _fmt_pct(n0_1, n0),
                c1      = _fmt_pct(n1_1, n1),
                smd     = _smd_binary(p0, p1, n0, n1),
                p       = _pvalue_str(pval),
                rtype   = "header",
            )

        if not present:
            continue

        _add_row(group_label, "", "", "", "", "", rtype="header")

        for v in present:
            n_all_1 = int((df[v] == 1).sum())
            n0_1    = int((g0[v] == 1).sum())
            n1_1    = int((g1[v] == 1).sum())
            p0, p1  = n0_1 / n0, n1_1 / n1

            tbl = np.array([[n1_1, n1 - n1_1], [n0_1, n0 - n0_1]])
            if tbl.min() < 5:
                _, pval = stats.fisher_exact(tbl)
            else:
                _, pval, _, _ = stats.chi2_contingency(tbl)

            _add_row(
                label   = "  " + labels_map.get(v, v),
                overall = _fmt_pct(n_all_1, n_all),
                c0      = _fmt_pct(n0_1, n0),
                c1      = _fmt_pct(n1_1, n1),
                smd     = _smd_binary(p0, p1, n0, n1),
                p       = _pvalue_str(pval),
                rtype   = "binary",
            )

    result = pd.DataFrame(rows)

    # ── Move "Triage performed" right before the "Triage protocol questions" block ──
    triage_label = labels_map.get("triage", "triage")
    group_label  = "Triage protocol questions"
    var_values   = result["Variable"].tolist()
    if triage_label in var_values and group_label in var_values:
        triage_idx = var_values.index(triage_label)
        group_idx  = var_values.index(group_label)
        order = list(range(len(rows)))
        order.remove(triage_idx)
        order.insert(order.index(group_idx), triage_idx)
        result    = result.iloc[order].reset_index(drop=True)
        row_types = [row_types[i] for i in order]

    # ── 1. Console print ─────────────────────────────────────────────────────
    print(display_title)
    print("=" * len(display_title))
    print(result.to_string(index=False))
    print("\nNumeric: median [Q1, Q3] | Mann-Whitney U test.")
    print("Categorical: N (%) per value | omnibus chi-square test (p-value on header row).")
    print("Binary / grouped: N (%) | Chi-square or Fisher's exact test.")
    print("SMD = standardised mean difference [95% CI].")

    # ── 2. Word export ───────────────────────────────────────────────────────
    doc   = Document()
    style = doc.styles['Normal']
    style.font.name = 'Calibri'
    style.font.size = Pt(10)

    tp = doc.add_paragraph()
    tr = tp.add_run(display_title)
    tr.bold = True
    tr.font.size = Pt(11)
    doc.add_paragraph()

    n_cols = len(result.columns)
    table  = doc.add_table(rows=len(result) + 1, cols=n_cols)
    table.style = 'Table Grid'

    HDR_COLOR   = RGBColor(0x4F, 0x81, 0xBD)
    GROUP_COLOR = RGBColor(0x9D, 0xC3, 0xE6)
    ROW_COLOR   = RGBColor(0xDC, 0xE6, 0xF1)

    def _shd(cell, rgb):
        tc   = cell._tc
        tcPr = tc.get_or_add_tcPr()
        shd  = OxmlElement('w:shd')
        shd.set(qn('w:val'),   'clear')
        shd.set(qn('w:color'), 'auto')
        shd.set(qn('w:fill'),  f"{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}")
        tcPr.append(shd)

    for i, col_name in enumerate(result.columns):
        cell = table.rows[0].cells[i]
        cell.text = col_name
        run = cell.paragraphs[0].runs[0]
        run.bold = True
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        run.font.size = Pt(9)
        _shd(cell, HDR_COLOR)

    for ri, (_, row_data) in enumerate(result.iterrows()):
        rtype = row_types[ri]
        cells = table.rows[ri + 1].cells
        for ci, value in enumerate(row_data):
            cells[ci].text = str(value)
            run = cells[ci].paragraphs[0].runs[0]
            run.font.size = Pt(9)
            if rtype == "header":
                run.bold = True
                _shd(cells[ci], GROUP_COLOR)
            else:
                _shd(cells[ci], ROW_COLOR)

    doc.add_paragraph()
    note = doc.add_paragraph()
    nr = note.add_run(
        "Numeric variables: median [Q1, Q3]; Mann–Whitney U test. "
        "Categorical variables: N (%) per value; omnibus chi-square test. "
        "Binary / grouped variables: N (%); Chi-square or Fisher's exact test. "
        "SMD = standardised mean difference [95% CI]."
    )
    nr.italic = True
    nr.font.size = Pt(9)

    suffix    = f"_{subtable_name}" if subtable_name else ""
    word_path = os.path.join(output_dir, f"table1_stroke{suffix}.docx")
    doc.save(word_path)
    print(f"\n[Saved] Word → {word_path}")

    # ── 3. JPG export ────────────────────────────────────────────────────────
    fig_h = max(3, len(result) * 0.45 + 1.5)
    fig, ax = plt.subplots(figsize=(17, fig_h))
    ax.axis("off")

    mpl_table = ax.table(
        cellText  = result.values,
        colLabels = result.columns,
        cellLoc   = "center",
        loc       = "center",
    )
    mpl_table.auto_set_font_size(False)
    mpl_table.set_fontsize(8)
    mpl_table.auto_set_column_width(col=list(range(len(result.columns))))

    for (r, c), cell in mpl_table.get_celld().items():
        cell.set_edgecolor("#AAAAAA")
        if r == 0:
            cell.set_facecolor("#4F81BD")
            cell.set_text_props(weight="bold", color="white")
        elif row_types[r - 1] == "header":
            cell.set_facecolor("#9DC3E6")
            cell.set_text_props(weight="bold", color="black")
        else:
            cell.set_facecolor("#DCE6F1")
            cell.set_text_props(color="black")

    fig.suptitle(display_title, fontsize=9, weight="bold", y=0.99)
    plt.tight_layout()

    jpg_path = os.path.join(output_dir, f"table1_stroke{suffix}.jpg")
    fig.savefig(jpg_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[Saved] JPG  → {jpg_path}")

    return result

# Tables

## General - p1 real

In [ ]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
table1 = build_table1(
    data_path        = DATA_PATH,
    outcome          = "p1_real_emerg",
    outcome_label    = "P1 (real priority)",
    vars_numeric     = VARS_NUMERIC,
    vars_categorical = VARS_CATEGORICAL,
    vars_binary      = VARS_BINARY,
    groups           = GROUPS,
    labels_map       = LABELS_MAP,
    title            = TITLE,
    output_dir       = OUTPUT_DIR,
    subtable_name    = SUBTABLE_NAME,
    table1_filtered  = TABLE1_FILTERED,
    years_filtered   = YEARS_FILTERED,
    vars_concrete    = VARS_CONCRETE,
)


## COVID period - p1 real

In [ ]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
table1 = build_table1(
    data_path        = DATA_PATH,
    outcome          = "p1_real_emerg",
    outcome_label    = "P1 (real emergency)",
    vars_numeric     = VARS_NUMERIC,
    vars_categorical = VARS_CATEGORICAL,
    vars_binary      = VARS_BINARY,
    groups           = GROUPS,
    labels_map       = LABELS_MAP,
    title            = TITLE,
    output_dir       = OUTPUT_DIR,
    subtable_name    = "covid_period_p1real",
    table1_filtered  = True,
    years_filtered   = ("2020-03-15", "2021-12-31"),
    vars_concrete    = ["age", "sex"]
)

## post-COVID period - p1 real

In [ ]:
table1 = build_table1(
    data_path        = DATA_PATH,
    outcome          = "p1_real_emerg",
    outcome_label    = "P1 (real emergency)",
    vars_numeric     = VARS_NUMERIC,
    vars_categorical = VARS_CATEGORICAL,
    vars_binary      = VARS_BINARY,
    groups           = GROUPS,
    labels_map       = LABELS_MAP,
    title            = TITLE,
    output_dir       = OUTPUT_DIR,
    subtable_name    = "postcovid_period_p1real",
    table1_filtered  = True,
    years_filtered   = ("2022-01-01", "2024-12-31"),
    vars_concrete    = ["age", "sex"]
)

## General - p1_assigned

In [ ]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
table1 = build_table1(
    data_path        = DATA_PATH,
    outcome          = "p1_assigned",
    outcome_label    = "P1 (assigned)",
    vars_numeric     = VARS_NUMERIC,
    vars_categorical = VARS_CATEGORICAL,
    vars_binary      = VARS_BINARY,
    groups           = GROUPS,
    labels_map       = LABELS_MAP,
    title            = TITLE,
    output_dir       = OUTPUT_DIR,
    subtable_name    = "all_period_p1assigned",
    table1_filtered  = True,
    years_filtered   = ("2020-01-01", "2024-12-31"),
    vars_concrete    = ["age", "sex", "triage", "p1_real_emerg", "has_icd_emerg"],
    df_input         = df_stroke_p1_assigned,
)

## COVID period - p1 assigned

In [ ]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
table1 = build_table1(
    data_path        = DATA_PATH,
    outcome          = "p1_assigned",
    outcome_label    = "P1 (assigned)",
    vars_numeric     = VARS_NUMERIC,
    vars_categorical = VARS_CATEGORICAL,
    vars_binary      = VARS_BINARY,
    groups           = GROUPS,
    labels_map       = LABELS_MAP,
    title            = TITLE,
    output_dir       = OUTPUT_DIR,
    subtable_name    = "covid_period_p1assigned",
    table1_filtered  = True,
    years_filtered   = ("2020-03-15", "2021-12-31"),
    vars_concrete    = ["age", "sex", "triage", "p1_real_emerg", "has_icd_emerg"],
    df_input         = df_stroke_p1_assigned,
)

## post-COVID period - p1 assigned

In [ ]:
table1 = build_table1(
    data_path        = DATA_PATH,
    outcome          = "p1_assigned",
    outcome_label    = "P1 (assigned)",
    vars_numeric     = VARS_NUMERIC,
    vars_categorical = VARS_CATEGORICAL,
    vars_binary      = VARS_BINARY,
    groups           = GROUPS,
    labels_map       = LABELS_MAP,
    title            = TITLE,
    output_dir       = OUTPUT_DIR,
    subtable_name    = "postcovid_period_p1assigned",
    table1_filtered  = True,
    years_filtered   = ("2022-01-01", "2024-12-31"),
    vars_concrete    = ["age", "sex", "triage", "p1_real_emerg", "has_icd_emerg"],
    df_input         = df_stroke_p1_assigned,
)